In [3]:
# ─────────────────────────────────────────────────────────
# CONFIG ← modifie ici
# ─────────────────────────────────────────────────────────
CSV_PATH = "comparaison_geocodage.csv"
# ─────────────────────────────────────────────────────────

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

df = pd.read_csv(CSV_PATH)

COLORS = {
    "nominatim": "#378ADD",
    "photon":    "#1D9E75",
    "arcgis":    "#D85A30",
}

# ─────────────────────────────────────────────────────────
# 1. Métriques résumées
# ─────────────────────────────────────────────────────────
print("=" * 55)
print(f"  {'Source':<12} {'Médiane':>10} {'Moyenne':>10} {'Max':>12}")
print("=" * 55)
for s in ["nominatim", "photon", "arcgis"]:
    ok = df[df[f"ok_{s}"] == True][f"dist_{s}_m"].dropna()
    print(f"  {s:<12} {ok.median():>9.0f}m {ok.mean():>9.0f}m {ok.max():>11.0f}m")
print("=" * 55)


# ─────────────────────────────────────────────────────────
# 2. Distribution cumulée
# ─────────────────────────────────────────────────────────
seuils = [50, 100, 500, 1_000, 5_000, 10_000, 50_000, 100_000]
labels = ["50m", "100m", "500m", "1km", "5km", "10km", "50km", "100km"]

fig_dist = go.Figure()
for s in ["nominatim", "photon", "arcgis"]:
    ok = df[df[f"ok_{s}"] == True][f"dist_{s}_m"].dropna()
    pcts = [(ok < seuil).mean() * 100 for seuil in seuils]
    fig_dist.add_trace(go.Scatter(
        x=labels, y=pcts,
        mode="lines+markers",
        name=s.capitalize(),
        line=dict(color=COLORS[s], width=2),
        marker=dict(size=6),
    ))

fig_dist.update_layout(
    title="Distribution cumulée des écarts vs CSV data.gouv",
    xaxis_title="Seuil de distance",
    yaxis_title="% communes en-dessous du seuil",
    yaxis=dict(range=[0, 100], ticksuffix="%"),
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
    height=420,
)
fig_dist.show()


# ─────────────────────────────────────────────────────────
# 3. Histogramme des écarts (log) — trois services superposés
# ─────────────────────────────────────────────────────────
fig_hist = go.Figure()
for s in ["nominatim", "photon", "arcgis"]:
    ok = df[df[f"ok_{s}"] == True][f"dist_{s}_m"].dropna()
    # On plafonne à 50 km pour la lisibilité
    vals = np.log10(ok.clip(upper=50_000) + 1)
    fig_hist.add_trace(go.Histogram(
        x=vals,
        name=s.capitalize(),
        opacity=0.6,
        marker_color=COLORS[s],
        nbinsx=50,
    ))

tickvals = [0, 1, 2, 3, 4, 4.7]
ticktext = ["1m", "10m", "100m", "1km", "10km", "50km+"]

fig_hist.update_layout(
    barmode="overlay",
    title="Distribution des écarts (échelle log)",
    xaxis=dict(title="Distance vs CSV", tickvals=tickvals, ticktext=ticktext),
    yaxis_title="Nombre de communes",
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
    height=400,
)
fig_hist.show()


# ─────────────────────────────────────────────────────────
# 4. Scatter Nominatim vs ArcGIS (log-log)
# ─────────────────────────────────────────────────────────
common = df[df["ok_nominatim"] & df["ok_arcgis"]].dropna(
    subset=["dist_nominatim_m", "dist_arcgis_m"]
).copy()

common["diverge"] = (common["dist_nominatim_m"] - common["dist_arcgis_m"]).abs() > 10_000

fig_scatter = go.Figure()

for diverge, label, color, size in [(False, "Accord", "#378ADD", 6), (True, "Divergence forte", "#E24B4A", 9)]:
    sub = common[common["diverge"] == diverge]
    fig_scatter.add_trace(go.Scatter(
        x=sub["dist_nominatim_m"],
        y=sub["dist_arcgis_m"],
        mode="markers",
        name=label,
        marker=dict(color=color, size=size, opacity=0.6),
        text=sub["nom_standard"] + " (" + sub["dep_nom"] + ")",
        hovertemplate="%{text}<br>Nominatim: %{x:.0f} m<br>ArcGIS: %{y:.0f} m<extra></extra>",
    ))

# Ligne diagonale y=x
maxval = max(common["dist_nominatim_m"].max(), common["dist_arcgis_m"].max())
fig_scatter.add_trace(go.Scatter(
    x=[1, maxval], y=[1, maxval],
    mode="lines",
    line=dict(color="gray", dash="dash", width=1),
    showlegend=False,
))

fig_scatter.update_layout(
    title="Nominatim vs ArcGIS — écart vs CSV par commune",
    xaxis=dict(title="Nominatim vs CSV (m)", type="log"),
    yaxis=dict(title="ArcGIS vs CSV (m)", type="log"),
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
    height=500,
)
fig_scatter.show()


# ─────────────────────────────────────────────────────────
# 5. Carte — communes avec les plus grands écarts
# ─────────────────────────────────────────────────────────
# On prend les communes avec au moins un service > 50 km
mask = (
    (df["dist_nominatim_m"] > 50_000) |
    (df["dist_photon_m"]    > 50_000) |
    (df["dist_arcgis_m"]    > 50_000)
)
outliers = df[mask].copy()
outliers["max_ecart_km"] = outliers[
    ["dist_nominatim_m", "dist_photon_m", "dist_arcgis_m"]
].max(axis=1) / 1000
outliers["info"] = (
    outliers["nom_standard"] + " (" + outliers["dep_nom"] + ")<br>"
    + "Nominatim: " + outliers["dist_nominatim_m"].fillna(0).div(1000).round(1).astype(str) + " km<br>"
    + "Photon:    " + outliers["dist_photon_m"].fillna(0).div(1000).round(1).astype(str) + " km<br>"
    + "ArcGIS:    " + outliers["dist_arcgis_m"].fillna(0).div(1000).round(1).astype(str) + " km"
)

fig_map = px.scatter_map(
    outliers,
    lat="latitude_centre",
    lon="longitude_centre",
    size="max_ecart_km",
    color="max_ecart_km",
    hover_name="nom_standard",
    hover_data={"info": True, "max_ecart_km": ":.0f", "latitude_centre": False, "longitude_centre": False},
    color_continuous_scale="OrRd",
    size_max=25,
    zoom=5,
    center={"lat": 46.5, "lon": 2.5},
    title="Communes avec le plus grand écart (au moins un service > 50 km)",
)
fig_map.update_layout(height=550, coloraxis_colorbar=dict(title="Écart max (km)"))
fig_map.show()


# ─────────────────────────────────────────────────────────
# 6. Tableau des cas extrêmes
# ─────────────────────────────────────────────────────────
print("\nTop 15 communes — plus grand écart Nominatim vs CSV :")
top = df.nlargest(15, "dist_nominatim_m")[[
    "nom_standard", "dep_nom",
    "dist_nominatim_m", "dist_photon_m", "dist_arcgis_m"
]].copy()
for col in ["dist_nominatim_m", "dist_photon_m", "dist_arcgis_m"]:
    top[col] = (top[col] / 1000).round(1).astype(str) + " km"
top.columns = ["Commune", "Département", "Nominatim", "Photon", "ArcGIS"]
print(top.to_string(index=False))

  Source          Médiane    Moyenne          Max
  nominatim          822m     20465m      790393m
  photon             873m     33136m      665295m
  arcgis             804m     19239m      790223m



Top 15 communes — plus grand écart Nominatim vs CSV :
      Commune             Département Nominatim   Photon   ArcGIS
      Lagarde         Hautes-Pyrénées  790.4 km 254.0 km 790.2 km
 Saint-Gorgon                Morbihan  665.3 km 665.3 km   0.8 km
Saint-Nazaire                    Gard  630.7 km 372.2 km 630.7 km
      Rumilly            Haute-Savoie  599.9 km 599.9 km   1.2 km
  Saint-Simon                     Lot  569.9 km 193.3 km  59.8 km
        Thèze Alpes-de-Haute-Provence  511.3 km 511.3 km 511.4 km
         Doux             Deux-Sèvres  449.8 km 194.6 km 449.7 km
    Les Loges             Haute-Marne  437.4 km 270.8 km 437.4 km
   Chevrières                   Loire  437.4 km  87.1 km 437.5 km
          Igé                    Orne  382.8 km 382.8 km 382.8 km
   Saint-Léon                  Allier  374.2 km   0.8 km 374.2 km
   Désertines                 Mayenne  352.8 km 352.8 km 353.0 km
       Lasson                   Yonne  340.3 km   0.7 km   0.8 km
      Buxeuil        